In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# =========================================================
# === 1️⃣ 数据加载
# =========================================================

obs_ds = xr.open_dataset("/home/users/kastanie/Global_regional_analysis/atmos-WRAF01-v4-1/hottestmonth/Berkeley_hottest_month_by_region_revised_by_ERA5.nc")
mask_ds = xr.open_dataset("/home/users/kastanie/Global_regional_analysis/atmos-WRAF01-v4-1/region_mask_1deg.nc")

region_mask = mask_ds["region_id"]
obs_data = obs_ds["t2m_hottest"].sel(year=slice(1981, 2024))

# ENSO monthly
nino_ds = xr.open_dataset("nino34_monthly_1980_2024.nc")
nino = nino_ds["nino34"]

# =========================================================
# === 2️⃣ 构造 (year, month) ENSO index
# =========================================================

nino_ym = nino.groupby("time.year").apply(
    lambda x: x.groupby("time.month").mean()
)

# rename 维度
nino_ym = nino_ym.rename({"year": "year", "month": "month"})

# 只保留 1981–2024
nino_ym = nino_ym.sel(year=slice(1981, 2024))

print(nino_ym.shape)
# =========================================================
# === 3️⃣ 逐区域：找最大相关系数 & 对应月份
# =========================================================

regions = obs_data.region_id.values

max_corr = np.full(len(regions), np.nan)
best_month = np.full(len(regions), np.nan)

common_years = np.intersect1d(obs_data.year.values, nino_ym.year.values)

obs_sel = obs_data.sel(year=common_years)
nino_sel = nino_ym.sel(year=common_years)

for j, r in enumerate(regions):

    y = obs_sel.sel(region_id=r).values

    corr_list = []

    for m in months:

        x = nino_sel.sel(month=m).values

        mask = (~np.isnan(x)) & (~np.isnan(y))

        if mask.sum() < 10:
            corr = np.nan
        else:
            corr, _ = pearsonr(x[mask], y[mask])

        corr_list.append(corr)

    corr_arr = np.array(corr_list)

    if np.all(np.isnan(corr_arr)):
        continue

    idx = np.nanargmax(np.abs(corr_arr))

    max_corr[j] = corr_arr[idx]
    best_month[j] = months[idx]

# 转 DataArray
max_corr_da = xr.DataArray(
    max_corr,
    coords={"region_id": regions},
    dims=["region_id"]
)

best_month_da = xr.DataArray(
    best_month,
    coords={"region_id": regions},
    dims=["region_id"]
)

# ====================================================
# === 4️⃣ 映射到 grid
# ====================================================

corr_map = xr.full_like(region_mask, np.nan, dtype=float)
month_map = xr.full_like(region_mask, np.nan, dtype=float)

for rid in regions:

    corr_map = xr.where(
        region_mask == rid,
        max_corr_da.sel(region_id=rid),
        corr_map
    )

    month_map = xr.where(
        region_mask == rid,
        best_month_da.sel(region_id=rid),
        month_map
    )

# ====================================================
# === 5️⃣ 画图
# ====================================================

proj = ccrs.PlateCarree()

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 5),
    subplot_kw=dict(projection=proj)
)

# ---- 图1：相关系数 ----
levels_corr = np.linspace(-0.6, 0.6, 21)

im1 = corr_map.plot(
    ax=axes[0],
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    levels=levels_corr,
    add_colorbar=False,
    extend = "both"
)

axes[0].set_title("Max ENSO correlation (OBS)")
axes[0].coastlines(linewidth=0.6)
axes[0].add_feature(cfeature.BORDERS, linewidth=0.3)
axes[0].set_extent([-180, 180, -60, 90])

# ---- 图2：月份 ----
levels_month = np.arange(1, 14)

im2 = month_map.plot(
    ax=axes[1],
    transform=ccrs.PlateCarree(),
    cmap="Paired",
    levels=levels_month,
    add_colorbar=False,
    extend = "both"
)

axes[1].set_title("Best ENSO month (1–12)")
axes[1].coastlines(linewidth=0.6)
axes[1].add_feature(cfeature.BORDERS, linewidth=0.3)
axes[1].set_extent([-180, 180, -60, 90])

# ---- colorbars ----
cbar_ax1 = fig.add_axes([0.1, 0.1, 0.35, 0.02])
cbar1 = fig.colorbar(im1, cax=cbar_ax1, orientation="horizontal")
cbar1.set_label("Correlation")

cbar_ax2 = fig.add_axes([0.55, 0.1, 0.35, 0.02])
cbar2 = fig.colorbar(im2, cax=cbar_ax2, orientation="horizontal")
cbar2.set_label("Month")

plt.savefig("ENSO_monthly_max_correlation_map.png", dpi=300, bbox_inches='tight')
plt.show()

print("✅ Finished: Monthly ENSO correlation analysis.")

# ====================================================
# === 6️⃣ 构造 region-specific ENSO time series
# ====================================================

years = nino_sel.year.values
n_years = len(years)

enso_region_array = np.full((len(regions), n_years), np.nan)

for j, r in enumerate(regions):

    m = best_month_da.sel(region_id=r).values

    if np.isnan(m):
        continue

    m = int(m)

    # 取该月份的 ENSO 时间序列
    enso_ts = nino_sel.sel(month=m).values  # (year,)

    enso_region_array[j, :] = enso_ts

# 转为 DataArray
enso_region_da = xr.DataArray(
    enso_region_array,
    coords={
        "region_id": regions,
        "year": years
    },
    dims=["region_id", "year"],
    name="nino34_best_month"
)

# ====================================================
# === 7️⃣ 保存为 nc 文件
# ====================================================

output_ds = xr.Dataset({
    "nino34_best_month": enso_region_da,
    "best_month": best_month_da,
    "max_correlation": max_corr_da
})

output_ds.to_netcdf("ENSO_best_month_timeseries_by_region.nc")

print("✅ Saved: ENSO_best_month_timeseries_by_region.nc")

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

# =====================================================
# 1. 读取数据
# =====================================================

ds = xr.open_dataset("SST_IFS_4season.nc")

# 只取前25个member
ds = ds.isel(number=slice(0,25))

sst = ds["sst"]

numbers = sst.number.values
lats = sst.latitude.values
lons = sst.longitude.values

# =====================================================
# 2. 创建时间容器
# =====================================================

years = np.arange(1981, 2026)   # 1981–2025
months = np.arange(1,13)

new_sst = np.full(
    (len(numbers), len(years), len(months), len(lats), len(lons)),
    np.nan,
    dtype=np.float32
)

# =====================================================
# 3. 重构时间
# =====================================================

frt = ds.forecast_reference_time.values
lead = ds.forecastMonth.values
print(lead)
for i, ref_time in enumerate(frt):

    ref_date = pd.to_datetime(ref_time)
    ref_year = ref_date.year
    ref_month = ref_date.month

    for j, lead_m in enumerate(lead):

        # 实际月份
        target_month = ref_month + lead_m
        target_year = ref_year

        if target_month > 12:
            #target_month -= 12
            target_month = target_month
            #target_year += 1

        # 超出范围就跳过
        if target_year < 1981 or target_year > 2025:
            continue
        year_index = target_year - 1981
        month_index = target_month - 3

        new_sst[:, year_index, month_index, :, :] = sst[:, i, j, :, :].values


# =====================================================
# 4. 构建新的 Dataset
# =====================================================

new_ds = xr.Dataset(
    {
        "sst": (["number","year","month","latitude","longitude"], new_sst)
    },
    coords={
        "number": numbers,
        "year": years,
        "month": months,
        "latitude": lats,
        "longitude": lons
    }
)

# =====================================================
# 5. 保存文件
# =====================================================

new_ds.to_netcdf("SST_IFS_monthly_reconstructed.nc")

print("Finished!")

In [ ]:
import numpy as np
import xarray as xr

# =========================================================
# 1️⃣ 读取数据
# =========================================================
ds = xr.open_dataset("SST_IFS_monthly_reconstructed.nc")
sst = ds["sst"]

# =========================================================
# 2️⃣ 选 Nino3.4 区域
# (你这个数据已经是 -179.5 → -110.5，不需要转换经度)
# =========================================================
nino = sst.sel(
    latitude=slice(5, -5),   # 注意纬度是降序
    longitude=slice(-170, -120)
)

# =========================================================
# 3️⃣ 计算 climatology（逐成员！！关键点）
# =========================================================
# 选基准期
nino_base = nino.sel(year=slice(1981, 2010))

# climatology（逐 member）
clim = nino_base.groupby("month").mean("year")
print(clim.shape)
print(nino.shape)

# =========================================================
# 4️⃣ 计算 anomaly
# =========================================================
anom = nino.groupby("month") - clim
print(anom.shape)
# =========================================================
# 5️⃣ 纬度加权平均（cos(lat)）
# =========================================================
weights = np.cos(np.deg2rad(anom.latitude))
weights.name = "weights"

#nino34 = anom.weighted(weights).mean(("latitude", "longitude"))

nino34 = anom.mean(("latitude", "longitude"))
# =========================================================
# 7️⃣ 保存结果
# =========================================================
nino34.name = "nino34_index"

nino34.to_netcdf("IFS_nino34_monthly.nc")

In [ ]:
#先处理然后保存每个区域的最相关月份的nino3.4指数

import numpy as np
import xarray as xr
from scipy.stats import pearsonr

# =========================================================
# === 1️⃣ 数据加载
# =========================================================

ifs_ds = xr.open_dataset("/home/users/kastanie/Global_regional_analysis/atmos-WRAF01-v4-1/hottestmonth/IFS_t2m_hottest_month.nc")
ifs_data = ifs_ds["t2m_hottest_month"].sel(year=slice(1981, 2024))

members = ifs_data.number.values
regions = ifs_data.region_id.values
years = ifs_data.year.values

# ENSO
nino_ds = xr.open_dataset("IFS_nino34_monthly.nc")
nino = nino_ds["nino34_index"]

# =========================================================
# === 3️⃣ 对齐年份
# =========================================================

common_years = np.intersect1d(years, nino.year.values)

ifs_sel = ifs_data.sel(year=common_years)
nino_sel = nino.sel(year=common_years)

years = common_years
nyear = len(years)

# =========================================================
# === 4️⃣ 计算 best month + 构造新 ENSO index
# =========================================================

# 输出数组
best_month_all = np.full((len(members), len(regions)), np.nan)
nino_new = np.full((len(members), len(regions), nyear), np.nan)

for i, m in enumerate(members):

    print(f"Processing member {m}")

    for j, r in enumerate(regions):

        # IFS 时间序列
        y = ifs_sel.sel(number=m, region_id=r).values

        corr_list = []

        # 12个月逐一算相关
        for mo in months:

            x = nino_sel.sel(month=mo, number=m).values

            mask = (~np.isnan(x)) & (~np.isnan(y))

            if mask.sum() < 10:
                corr = np.nan
            else:
                corr, _ = pearsonr(x[mask], y[mask])

            corr_list.append(corr)

        corr_arr = np.array(corr_list)

        if np.all(np.isnan(corr_arr)):
            continue

        # 找最大绝对值相关
        idx = np.nanargmax(np.abs(corr_arr))
        best_mo = months[idx]

        best_month_all[i, j] = best_mo

        # =================================================
        # ⭐ 核心：提取该月的 ENSO 序列
        # =================================================
        best_nino_series = nino_sel.sel(month=best_mo).values  # (year)

        nino_new[:, j, :] = best_nino_series

# =========================================================
# === 5️⃣ 转为 xarray
# =========================================================

nino_new_da = xr.DataArray(
    nino_new,
    coords={
        "number": members,
        "region_id": regions,
        "year": years
    },
    dims=["number", "region_id", "year"],
    name="nino34_optimal"
)

best_month_da = xr.DataArray(
    best_month_all,
    coords={
        "number": members,
        "region_id": regions
    },
    dims=["number", "region_id"],
    name="best_month"
)

# =========================================================
# === 6️⃣ 保存
# =========================================================

ds_out = xr.Dataset({
    "nino34_optimal": nino_new_da,
    "best_month": best_month_da
})

ds_out.to_netcdf("nino34_optimal_matched_to_IFS.nc")

print("✅ Finished: optimal ENSO index constructed.")

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import genextreme as gev
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import BoundaryNorm
from matplotlib.gridspec import GridSpec

# =========================================================
# === 加载数据
# =========================================================

ecmwf_ds = xr.open_dataset("/home/users/kastanie/Global_regional_analysis/atmos-WRAF01-v4-1/hottestmonth/IFS_t2m_hottest_month.nc")
rl_ds = xr.open_dataset("/home/users/kastanie/Global_regional_analysis/atmos-WRAF01-v4-1/hottestmonth/Berkeley_hottest_month_by_region_revised_by_ERA5.nc")
mask_ds = xr.open_dataset("/home/users/kastanie/Global_regional_analysis/atmos-WRAF01-v4-1/region_mask_1deg.nc")
fidelity_ds = xr.open_dataset("/home/users/kastanie/Global_regional_analysis/atmos-WRAF01-v4-1/hottestmonth/grid_comparison_fidelity_trend_corrected.nc")

# =========================================================
# === 🔴 ENSO：先对 ensemble 平均
# =========================================================

nino_ds = xr.open_dataset("nino34_optimal_matched_to_IFS.nc")

# (number, region_id, year) → (region_id, year)
nino_full = nino_ds["nino34_optimal"]
print(nino_full)
nino_ds = xr.open_dataset("ENSO_best_month_timeseries_by_region.nc")
nino = nino_ds["nino34_best_month"]
# =========================================================
# === ENSO 去除函数（🔴改成按 region 用不同 index）
# =========================================================
import numpy as np
import xarray as xr

def remove_enso_signal(data, nino_index):

    data = data.copy()

    common_years = np.intersect1d(data.year.values, nino_index.year.values)

    data_sel = data.sel(year=common_years)
    nino_sel = nino_index.sel(year=common_years)

    # 判断是否有 ensemble 维度
    has_member = "number" in data.dims

    for r in data.region_id.values:

        if not has_member:

            y = data_sel.sel(region_id=r).values
            x = nino_sel.sel(region_id=r).values

            mask = (~np.isnan(y)) & (~np.isnan(x))

            if mask.sum() < 10:
                continue

            beta, alpha = np.polyfit(x[mask], y[mask], 1)

            enso_component = beta * nino_sel.sel(region_id=r)

            data.loc[dict(region_id=r, year=common_years)] = (
                data_sel.sel(region_id=r) - enso_component
            )

        else:

            for m in data.number.values:

                y = data_sel.sel(number=m, region_id=r).values
                x = nino_sel.sel(number=m, region_id=r).values

                mask = (~np.isnan(y)) & (~np.isnan(x))

                if mask.sum() < 10:
                    continue

                beta, alpha = np.polyfit(x[mask], y[mask], 1)

                enso_component = beta * nino_sel.sel(number=m, region_id=r)

                data.loc[dict(number=m, region_id=r, year=common_years)] = (
                    data_sel.sel(number=m, region_id=r) - enso_component
                )

    return data
# =========================================================
# === 1️⃣ 只对 Berkeley 去 ENSO
# =========================================================

rl_enso_removed = remove_enso_signal(
    rl_ds["t2m_hottest"].sel(year=slice(1981, 2024)),
    nino
)

rl_full = rl_ds["t2m_hottest"].copy()
rl_full.loc[dict(year=slice(1981, 2024))] = rl_enso_removed


# =========================================================
# === 2️⃣ IFS 去除每个成员中的集合平均部分
# =========================================================
ifs_subset = ecmwf_ds["t2m_hottest_month"].sel(year=slice(1981, 2024))

ifs_removed = remove_enso_signal(ifs_subset, nino_full)

ifs_full = ecmwf_ds["t2m_hottest_month"].copy()
ifs_full.loc[dict(year=slice(1981, 2024))] = ifs_removed

t2m_ifs_all = ifs_full
t2m_berkeley_all = rl_full


# =========================================================
# === 后续计算（完全不变）
# =========================================================

region_ids = mask_ds['region_id'].values
lat = mask_ds['latitude'].values
lon = mask_ds['longitude'].values

fidelity = fidelity_ds["fidelity_score"]
fidelity_mask = fidelity == 3
hatch_mask = (~fidelity_mask.values) & (~np.isnan(region_ids))

n_regions = 237

time_periods = [
    (1981, 1990),
    (1991, 2000),
    (2001, 2010),
    (2011, 2020),
    (2020, 2024)
]

n_periods = len(time_periods)


def get_max_before_year(ds, start, end):
    sub = ds.sel(year=(ds.year >= start) & (ds.year <= end))
    return sub.max(dim='year')


prob_map = np.full((n_regions, n_periods), np.nan)

for i, (start, end) in enumerate(time_periods):

    print(f"Processing {start}-{end}...")

    period_mask_ifs = (t2m_ifs_all.year >= start) & (t2m_ifs_all.year <= end)
    period_mask_berkeley = (t2m_berkeley_all.year >= start) & (t2m_berkeley_all.year <= end)

    t2m_ifs_period = t2m_ifs_all.sel(year=period_mask_ifs)
    t2m_berkeley_period = t2m_berkeley_all.sel(year=period_mask_berkeley)

    # bias correction
    ifs_mean = t2m_ifs_period.mean(dim='year')
    berkeley_mean = t2m_berkeley_period.mean(dim='year')

    t2m_corrected = t2m_ifs_period - ifs_mean + berkeley_mean
    
    # 历史最大（OBS 已去 ENSO）
    max_t2m = get_max_before_year(t2m_berkeley_all, 1940, start-1)

    values = t2m_corrected.values.transpose(0, 2, 1).reshape(-1, n_regions)

    for r in range(n_regions):

        series = values[:, r]
        series = series[~np.isnan(series)]

        if len(series) < 30:
            continue

        try:
            c, loc, scale = gev.fit(series)
            x = max_t2m.sel(region_id=r).item()
            F = gev.cdf(x, c, loc=loc, scale=scale)
            prob_map[r, i] = 1 - F
        except Exception:
            continue


# =========================================================
# === 保存
# =========================================================

period_labels = [f"{start}-{end}" for start, end in time_periods]

ds_out = xr.Dataset(
    {"record_breaking_probability": (("region_id", "period"), prob_map)},
    coords={"region_id": np.arange(n_regions), "period": period_labels}
)

#ds_out.to_netcdf("record_breaking_probability_removed_ENSO_ensemble_mean_from_OBS_and_IFS.nc")
ds_out.to_netcdf("record_breaking_probability_removed_ENSO_each_ensemble_from_IFS_and_ENSO_obs_from_obs.nc")

print("✅ Saved (OBS ENSO removed, region-dependent ENSO, IFS unchanged).")

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# =========================================================
# 洲 -> region_id
# =========================================================
continents = {
    "Asia": [70,108,109,110,111,112,113,114,115,116,117,118,119,120,121,
             122,123,124,125,126,127,130,133,135,136,137,138,139,140,141,
             142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,
             157,206,207,208,209,210,211,212,213,214,215,216,217,229,230,
             231,235,236],

    "Europe": [102,103,104,105,106,107,128,129,131,134,197,198,199,
               200,201,202,203,204,205,234],

    "Africa": [52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,
               69,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,
               87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,132,
               225,226,227,228],

    "North\nAmerica": list(range(0,32)) + [218,219,220,221,222,223,224,232],
    "South\nAmerica": list(range(32,52)) + [191,192,193,194,195,196,233],
    "Oceania": [158,159,160,161,162,163,164,165,166,167,168,169]
}

continent_names = list(continents.keys())

# =========================================================
# 时间段长度
# =========================================================
period_length = {
    "1981-1990": 10,
    "1991-2000": 10,
    "2001-2010": 10,
    "2011-2020": 10,
    "2020-2024": 5
}

# =========================================================
# 概率区间
# =========================================================
bins = np.array([0, 0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.25, 0.30, 10])
bin_labels = [
    "0-2%", "2-5%", "5-8%", "8-10%",
    "10-15%", "15-20%", "20-25%", "25-30%", ">30%"
]

# =========================================================
# 计算 summary
# =========================================================
def compute_summary(count_file, prob_file, periods_to_use):

    count_ds = xr.open_dataset(count_file)
    prob_ds = xr.open_dataset(prob_file)

    counts = count_ds["exceedance_count"]
    probs = prob_ds["record_breaking_probability"]

    periods = [p for p in prob_ds.period.values if p in periods_to_use]

    sum_counts = np.zeros((len(continent_names), len(bin_labels)))
    sum_region_years = np.zeros_like(sum_counts)

    for p in periods:

        p_length = period_length[p]

        for ci, cont in enumerate(continent_names):

            region_list = continents[cont]

            cont_prob = probs.sel(region_id=region_list, period=p)
            cont_count = counts.sel(region_id=region_list, period=p)

            bin_index = np.digitize(cont_prob.values, bins) - 1

            for bi in range(len(bin_labels)):

                mask = bin_index == bi
                n_bin_regions = mask.sum()

                if n_bin_regions == 0:
                    continue

                total_count = cont_count.values[mask].sum()

                sum_counts[ci, bi] += total_count
                sum_region_years[ci, bi] += n_bin_regions * p_length

    summary = sum_counts / sum_region_years
    summary[sum_region_years == 0] = np.nan

    return summary


# =========================================================
# 时间段
# =========================================================
periods_to_use = ["1981-1990", "1991-2000", "2001-2010", "2011-2020", "2020-2024"]

# =========================================================
# 计算三种情况
# =========================================================
summary_raw = compute_summary(
    "Berkeley_record_exceedance_counts.nc",
    "record_breaking_probability_by_region_and_period.nc",
    periods_to_use
)

summary_detrended = compute_summary(
    "Berkeley_record_exceedance_counts_detrended.nc",
    "record_breaking_probability_by_region_and_period_detrended.nc",
    periods_to_use
)

summary_enso = compute_summary(
    "Role_of_ENSO/Berkeley_record_exceedance_counts_ENSO_removed_by_obs_from_OBS.nc",
    "Role_of_ENSO/record_breaking_probability_removed_ENSO_each_ensemble_from_IFS_and_ENSO_obs_from_obs.nc",
    periods_to_use
)

# =========================================================
# 差值
# =========================================================
summary_diff_detrended = summary_detrended - summary_raw
summary_diff_enso = summary_enso - summary_raw

# =========================================================
# 绘图（带数值标注）
# =========================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

titles = [
    "Raw",
    "Detrended - Raw",
    "ENSO Removed - Raw"
]

data_list = [
    summary_raw,
    summary_diff_detrended,
    summary_diff_enso
]

panel_labels = ["(a)", "(b)", "(c)"]

for i, ax in enumerate(axes):

    if i == 0:
        cmap = "YlOrBr"
        vmin, vmax = 0, 0.35
        label = "Observed frequency"
    else:
        cmap = "RdBu_r"
        vmin, vmax = -0.1, 0.1
        label = "Difference" if i == 2 else " "

    data = data_list[i]

    im = ax.imshow(
        data,
        aspect="auto",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax
    )

    # =========================================================
    # ⭐ 在每个格子写数值
    # =========================================================
    for y in range(data.shape[0]):
        for x in range(data.shape[1]):

            value = data[y, x]

            if np.isnan(value):
                continue

            # 自动选择字体颜色（避免看不清）
            text_color = "white" if abs(value) > (vmax * 0.5) else "black"

            ax.text(
                x, y,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=12,
                color=text_color,
                rotation=90   # ⭐ 竖排
            )

    ax.set_xticks(range(len(bin_labels)))
    ax.set_xticklabels(bin_labels, rotation=45, fontsize=18)

    ax.set_yticks(range(len(continent_names)))
    ax.set_yticklabels(continent_names, fontsize=18)

    ax.set_title(titles[i], fontsize=22)

    ax.text(
        -0.20, 1.01, panel_labels[i],
        transform=ax.transAxes,
        fontsize=24,
        fontweight='bold',
        ha='left',
        va='bottom'
    )

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(label, fontsize=18)
    cbar.ax.tick_params(labelsize=18, length=4)

plt.tight_layout()

plt.savefig(
    "Fig5_continent_probability_3panels.pdf",
    dpi=300
)

plt.show()